# GSE149924 — targeted reanalysis of the autophagy claim

Mouse quadriceps, aged ± 15-PGDH inhibitor SW033291, 1 month, n=3 per arm (Palla et al. 2021, PMID 33303683).
Question under test: **does 15-PGDH inhibition increase autophagy in aged muscle?**

**How to read this notebook.** It is a faithful transcript of the code that produced every number in the
accompanying report, organised by item. It ran across two kernels — statistics in R 4.5.3
(`limma`, `edgeR`, `fgsea`) and figures in Python 3.11 (`matplotlib`, `pandas`, `scipy`) — so cells are
tagged `# --- R ---` or `# --- Python ---`. It is not a single-kernel one-click rerun; it has not been
re-executed end to end as one document.

Inputs are the two deposited DESeq2 tables. The third deposited file, `GSE149924_tpm.txt.gz`, is used
**only** in item 1 and is unusable for gene-level work (see item 1).


## Item 0 — inputs

In [ ]:
# --- Python --- fetch the deposited supplementary files
import subprocess
base = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE149nnn/GSE149924/suppl"
for f in ["GSE149924_tpm.txt.gz", "GSE149924_DESeq2_A_veh_Y_veh-2.txt.gz",
          "GSE149924_DEseq2_aged_SW_aged_Veh.csv.gz"]:
    subprocess.run(["curl","-sS","-O",f"{base}/{f}"], cwd="gse149924", check=True)
subprocess.run("gunzip -kf gse149924/*.gz", shell=True, check=True)

In [ ]:
# --- R --- load the two DESeq2 tables; BH-adjust the drug table (deposit carries raw p only)
d <- read.csv("gse149924/GSE149924_DEseq2_aged_SW_aged_Veh.csv", row.names=1)   # 24,013 rows
d$FDR_dep <- p.adjust(d$pvalue, "BH")
a <- read.delim("gse149924/GSE149924_DESeq2_A_veh_Y_veh-2.txt")                  # 813 rows, PRE-FILTERED
dd <- d[!is.na(d$stat) & d$baseMean > 10, ]        # 10,122 genes for set testing
st <- dd$stat; names(st) <- rownames(dd)
rk <- sort(setNames(d$stat[!is.na(d$stat)], rownames(d)[!is.na(d$stat)]), decreasing=TRUE)  # 16,675

## Item 1 (gate) — data integrity and design

**Verdict: gate partly failed.** The deposited TPM matrix is unusable; the design is confirmed and batch
is not the dominant axis.

In [ ]:
# --- R --- the TPM matrix's gene labels do not match its values
raw2 <- read.delim("gse149924/GSE149924_tpm.txt", check.names=FALSE, header=TRUE)
s2 <- raw2[[1]]; v2 <- rowMeans(as.matrix(raw2[,2:14]))
muscle <- c("Ckm","Mb","Acta1","Mylpf","Actn3")
rbind(TPM = round(v2[match(muscle, s2)]), deposited_baseMean = round(d[muscle, "baseMean"]))
cor(v2[match(rownames(d), s2)], d$baseMean, use="complete.obs", method="spearman")   # -0.017
# no row offset in +/-3000 recovers alignment (best Pearson 0.031) -> not a shift, a scramble

In [ ]:
# --- Python --- design QC. Sample covariance is invariant to row-label permutation and the rows are
# coherent (sample-sample Spearman 0.936-0.977), so PCA/clustering on the TPM matrix is still valid.
L = np.log2(num.values + 1)
rho, _ = spearmanr(L)
top = Ldf.loc[Ldf.var(axis=1).sort_values(ascending=False).index[:2000]]
X = top.sub(top.mean(axis=1), axis=0).T.values
U, S, Vt = np.linalg.svd(X, full_matrices=False); ve = S**2/np.sum(S**2)

def r2_perm(samples, labels):   # PERMANOVA-style R2 with exact permutation p
    sub = top[samples].sub(top[samples].mean(axis=1), axis=0).T.values
    D = squareform(pdist(sub)); n = len(samples)
    def ss(lab):
        lab = np.asarray(lab); tot = (D**2).sum()/(2*n); within = 0
        for g in set(lab):
            i = np.where(lab == g)[0]; within += (D[np.ix_(i,i)]**2).sum()/(2*len(i))
        return (tot - within)/tot
    obs = ss(labels); lab = np.asarray(labels); g1 = (lab == lab[0]).sum()
    null = [ss([lab[0] if i in c else 'x' for i in range(n)]) for c in combinations(range(n), g1)]
    return obs, np.mean(np.array(null) >= obs)
# NOTE: R2 must be read against its null expectation of 1/(n-1) for a two-group split, not against zero.

## Item 2 (gate) — cell composition

**Verdict: gate failed — uninformative rather than clean.** No marker shifts, but the median minimum
detectable effect across myeloid markers is 1.64 log2FC (3.1-fold), so a macrophage shift of the size
that would matter is not excluded. `Adgre1` is deposited under its former symbol `Emr1`.

In [ ]:
# --- R --- marker effect sizes with CIs and minimum detectable effect; myeloid set test
myeloid <- c("Cd68","Itgam","Ptprc","Lyz2","Mrc1","Cd163","Ctss","Ctsk","Emr1")
ci <- d[myeloid, c("baseMean","log2FoldChange","lfcSE","pvalue")]
ci$lo <- ci$log2FoldChange - 1.96*ci$lfcSE; ci$hi <- ci$log2FoldChange + 1.96*ci$lfcSE
ci$MDE_80pct <- 2.8*ci$lfcSE          # minimum detectable log2FC at 80% power, alpha = 0.05
gm <- intersect(myeloid, names(st)); obs <- mean(st[gm])
null <- replicate(20000, mean(sample(st, length(gm))))
(sum(abs(null) >= abs(obs)) + 1)/20001      # 0.60
# families the macrophage concern targets directly:
for (nm in c("^Cts","^Atp6v")) { g <- grep(nm, names(st), value=TRUE); print(mean(st[g])) }

## Item 3 (the decider) — FOXO-dependent autophagy vs atrogenes

**Verdict: weakens the hypothesis, and the control fails.** Custom sets, deliberately not stock GO terms.

In [ ]:
# --- R --- fgsea on the ranked Wald statistics; custom sets as specified
library(fgsea)
foxo_auto <- c("Map1lc3b","Gabarapl1","Bnip3","Bnip3l","Ctsl","Atg12","Atg4b","Ulk2","Pik3c3","Sqstm1")
atro      <- c("Fbxo32","Trim63","Fbxo30","Fbxo21","Traf6","Mstn")
set.seed(2024)
res <- fgsea(pathways=list(FOXO_autophagy=foxo_auto, FOXO_atrogene=atro),
             stats=rk, minSize=5, maxSize=500, nPermSimple=100000)
res[, c("pathway","size","ES","NES","pval","padj")]
#   FOXO_atrogene   NES -1.447  p 0.074   leading edge Traf6, Mstn, Fbxo30
#   FOXO_autophagy  NES -0.881  p 0.619   leading edge Ulk2, Ctsl, Bnip3l
d[c("Fbxo32","Trim63"), c("baseMean","log2FoldChange","lfcSE")]   # both flat, lfcSE 0.161
a["Fbxo32", c("log2FoldChange","padj")]                          # +0.614, padj 0.077 -> sign is correct

## Item 4 — CREB/TFEB bypass

**Verdict: neither supports nor falsifies — the route-is-live control fails.**

In [ ]:
# --- R --- CLEAR, non-FOXO core machinery, CREB targets, cAMP biosynthesis; TFEB family per gene
S <- list(
  CLEAR_lysosomal = c("Lamp1","Lamp2","Ctsa","Ctsb","Ctsd","Ctsf","Hexa","Hexb","Gba","Gusb","Npc1",
                      "Psap","Neu1","Mcoln1","Sgsh","Atp6v0a1","Atp6v0d1","Atp6v1a","Atp6v1h"),
  nonFOXO_core_autophagy = c("Ulk1","Atg13","Rb1cc1","Becn1","Atg14","Wipi1","Wipi2","Atg9a","Atg5",
                             "Atg7","Atg3","Atg16l1"),
  CREB_targets = c("Ppargc1a","Nr4a1","Nr4a2","Nr4a3","Fos","Jun","Crem","Sik1","Per1","Pde4b"),
  cAMP_biosynthesis_GO0006171 = camp)     # camp = QuickGO GO:0006171, mouse, descendants
set.seed(2025)
fgsea(pathways=S, stats=rk, minSize=5, maxSize=500, nPermSimple=100000)[, c("pathway","NES","pval")]
d[c("Tfeb","Tfe3","Mitf"), c("baseMean","log2FoldChange","lfcSE")]

## Item 5 — is the p62 readout confounded?

**Verdict: supports the hypothesis (weakly) — the simplest confound is excluded.**

In [ ]:
# --- R --- Sqstm1 directly; NRF2 targets; proteasome subunits
d["Sqstm1", c("baseMean","log2FoldChange","lfcSE")]      # -0.0016, lfcSE 0.121 -> CI +/-0.24
nrf2 <- c("Nqo1","Hmox1","Gclc","Gclm","Txnrd1","Gsta3","Srxn1")
psm  <- grep("^Psm[abcde]", rownames(d), value=TRUE)
psm  <- psm[d[psm,"baseMean"] > 10 & !is.na(d[psm,"stat"])]
set.seed(777)
fgsea(pathways=list(NRF2_targets=nrf2, Proteasome_subunits=psm), stats=rk,
      minSize=5, nPermSimple=100000)[, c("pathway","size","NES","pval","padj")]

## Item 6 — mitochondrial dynamics, and the internal positive control

**Verdict: weakens the fission alternative; the ETC control validates the whole series.**

In [ ]:
# --- R --- fission / fusion / mitophagy / biogenesis, then the ETC genes as internal control
M <- list(fission=c("Dnm1l","Fis1","Mff","Mief1","Mief2"), fusion=c("Mfn1","Mfn2","Opa1"),
          mitophagy_receptors=c("Pink1","Prkn","Fundc1","Bnip3","Bnip3l","Nipsnap1","Nipsnap2"),
          biogenesis=c("Tfam","Nrf1","Esrra","Ppargc1b","Polg","Polrmt"))
etc <- list(CI_Nduf=grep("^Nduf", names(st), value=TRUE), CII_Sdh=grep("^Sdh[abcd]$", names(st), value=TRUE),
            CIII_Uqcr=grep("^Uqcr", names(st), value=TRUE), CIV_Cox=grep("^Cox[0-9]|^Cox[45678]", names(st), value=TRUE),
            CV_Atp5=grep("^Atp5", names(st), value=TRUE))
etc$all_ETC <- unique(unlist(etc))
# expression-matched null, used for every set-level claim in this report:
dec <- cut(log10(dd$baseMean), breaks=quantile(log10(dd$baseMean), probs=seq(0,1,0.1)),
           include.lowest=TRUE, labels=FALSE); names(dec) <- rownames(dd)
matched_p <- function(g, B=10000) {
  tab <- table(dec[g]); obs <- mean(st[g])
  null <- replicate(B, { pick <- unlist(lapply(names(tab), function(k)
            sample(names(dec)[dec == as.integer(k)], tab[[k]]))); mean(st[pick]) })
  (sum(abs(null - mean(null)) >= abs(obs - mean(null))) + 1)/(B + 1)
}

## Item 7 — denervation programme

**Verdict: real but not specific — it does not replace the four-pathway model.**

In [ ]:
# --- R --- denervation set, then the specificity test against other age-raised genes
den <- c("Myog","Gadd45a","Runx1","Chrna1","Chrng","Ncam1","Musk","Myh3","Myh8")
g <- intersect(den, names(st)); mean(st[g]); matched_p(g)       # -0.902, p = 0.004
agup <- intersect(rownames(a)[a$log2FoldChange > 0 & !is.na(a$padj) & a$padj < 0.1], names(st))
wilcox.test(st[intersect(den, agup)], st[setdiff(agup, c(den,"Hdac4"))])$p.value    # 0.409
# and matched on aging effect-size quartile -> p = 0.612  (i.e. not distinguishable)

## Multiple-testing accounting

In [ ]:
# --- Python --- BH across the 32 distinct set-level tests run on the drug contrast in this session
from scipy.stats import false_discovery_control
F['BH'] = false_discovery_control(F.p.values, method='bh')
# denervation nominal 0.004 -> BH 0.016 ; autophagy regulation 1e-4 -> BH 5.3e-4 ; ETC 1e-4 -> 5.3e-4
# BH assumes independence or positive dependence; several of these sets overlap heavily, so treat as
# indicative rather than exact.

---
# Part 2 — Re-quantification from raw reads

Everything above uses the authors' deposited DESeq2 tables. This part rebuilds the drug contrast from the
raw FASTQs so that no step depends on their pipeline. Only the six batch-2 libraries (SRR11694853–858,
which constitute the entire SW-vs-vehicle comparison) were completed; the batch-1 libraries behind the
aging contrast were not retrieved, so the aged-vs-young comparison in Part 1 still rests on their
pre-filtered 813-gene list.

## 2.1 Library depth — why the drug arm is the weak one

In [ ]:
# --- Python --- read counts per run from ENA
import requests, pandas as pd, numpy as np
r = requests.get("https://www.ebi.ac.uk/ena/portal/api/filereport",
    params={"accession":"PRJNA630522","result":"read_run",
            "fields":"run_accession,sample_title,fastq_bytes,read_count,library_layout","format":"tsv"})
rows = [l.split('\t') for l in r.text.strip().splitlines()]
dep = pd.DataFrame(rows[1:], columns=rows[0]); dep['read_count'] = dep.read_count.astype(int)
dep['batch'] = np.where(dep.sample_title.str.contains('Batch 1'), 1, 2)
# batch 1 median 23.1 M pairs vs batch 2 median 5.9 M -> 3.9x; the drug contrast is entirely batch 2

## 2.2 Download, index, quantify

In [ ]:
# --- bash --- ENA serves ~0.2 MB/s single-stream and silently truncates; verify EVERY file.
#   curl -C - resume produced one CRC-corrupt gz and four 199-byte error stubs that exited 0.
#   NCBI's trace endpoint is ~5x faster but returns one concatenated stream, not split mates;
#   sra-tools prefetch could not resolve accessions from this sandbox.
# per-file fetch with verification and up to 3 attempts:
#   curl -sS -o "$f" "https://$u" && gunzip -t "$f" || retry
#
# salmon index (Ensembl GRCm39 release 112 cDNA + ncRNA, 145,918 transcripts -> 144,344 indexed):
# salmon index -t txome.fa -i salmon_idx -k 31 -p 8
#
# quantification, per run:
# salmon quant -i ref/salmon_idx -l A -1 ${r}_1.fastq.gz -2 ${r}_2.fastq.gz \
#     -p 8 --validateMappings --seqBias --gcBias -o quant/${r}
# mapping rates 78.5-83.4%; fragment counts match ENA read_count exactly.

## 2.3 tximport → DESeq2, and concordance with the deposited table

In [ ]:
# --- R ---
library(tximport); library(DESeq2)
t2g  <- read.delim("ref/tx2gene.tsv", header=FALSE, col.names=c("tx","gene_id","symbol"))
runs <- c(SRR11694853="Veh", SRR11694854="Veh", SRR11694855="Veh",
          SRR11694856="SW",  SRR11694857="SW",  SRR11694858="SW")
files <- file.path("quant", names(runs), "quant.sf"); names(files) <- names(runs)
txi  <- tximport(files, type="salmon", tx2gene=t2g[,c("tx","symbol")])
cd   <- data.frame(row.names=names(runs), condition=factor(unname(runs), levels=c("Veh","SW")))
dds  <- DESeqDataSetFromTximport(txi, cd, ~condition)
dds  <- DESeq(dds[rowSums(counts(dds)) >= 10, ], quiet=TRUE)
rs   <- results(dds, contrast=c("condition","SW","Veh"))
# 15,258 genes tested; 16 at FDR<0.05 (the deposit has 1); median log2FC +0.007, 49.3% down
# Spearman vs deposited log2FC = 0.783 over 9,422 shared genes -> the deposited drug table is sound

## 2.4 Set tests on independent counts, with an expression-matched null

In [ ]:
# --- R ---
nr  <- as.data.frame(rs); nr <- nr[!is.na(nr$stat) & nr$baseMean > 10, ]
st2 <- setNames(nr$stat, rownames(nr))
dec2 <- cut(log10(nr$baseMean), breaks=quantile(log10(nr$baseMean), probs=seq(0,1,0.1)),
            include.lowest=TRUE, labels=FALSE); names(dec2) <- rownames(nr)
matched_p2 <- function(g, B=10000) {       # null matched on baseMean decile
  g <- intersect(g, names(st2)); tab <- table(dec2[g]); obs <- mean(st2[g])
  null <- replicate(B, { pick <- unlist(lapply(names(tab), function(k)
            sample(names(dec2)[dec2 == as.integer(k)], tab[[k]]))); mean(st2[pick]) })
  c(obs, length(g), (sum(abs(null-mean(null)) >= abs(obs-mean(null)))+1)/(B+1)) }
# Result: every autophagy/lysosome set flat (BH > 0.47); ETC +1.616 (BH 0.0012).
# NOTE: the Part 1 'regulation of autophagy is down' finding ATTENUATES here
# (-0.301 -> -0.177, BH 0.136) and is RETRACTED. See docs/REPORT.md.

## 2.5 Per-sample composition score, and the myeloid-adjusted model

In [ ]:
# --- R --- this is the analysis the deposited tables could not support
vsd <- vst(dds, blind=TRUE); M <- assay(vsd)
mk  <- list(myeloid=c("Adgre1","Emr1","Cd68","Itgam","Ptprc","Lyz2","Mrc1","Cd163","Ctss","Ctsk"),
            endothelial=c("Pecam1","Cdh5","Kdr"), FAP=c("Pdgfra","Col1a1","Dcn"),
            myogenic=c("Pax7","Myod1","Myog"))
scores <- sapply(mk, function(g){ g <- intersect(g, rownames(M))
                 colMeans(t(scale(t(M[g, , drop=FALSE])))) })
# Adgre1 IS detected by salmon although absent from the deposit.
# myeloid score: Veh +0.173 vs SW -0.173 (diff -0.347, p = 0.60)
cd2 <- cd; cd2$myeloid <- scores[, "myeloid"]
dds2 <- DESeq(DESeqDataSetFromTximport(txi, cd2, ~ myeloid + condition), quiet=TRUE)
rs2  <- results(dds2, contrast=c("condition","SW","Veh"))
# Adjusted: CLEAR -0.027, lysosome -0.042, core machinery +0.060 -> still flat.
# ETC holds at +1.641. The composition gate is therefore closed.

## 2.6 What is still missing

The six batch-1 libraries (SRR11694846, 847, 849, 850, 851, 852) were not retrieved, so the aged-vs-young
contrast was never computed genome-wide. That limits two secondary questions only: whether the autophagy
sets move with age at all, and whether the denervation signature's non-specificity survives comparison
against a complete gene list rather than the authors' filtered 813.

## 2.7 Per-sample heatmap

In [ ]:
# --- R --- variance-stabilised matrix, z-scored within gene
vsd <- vst(dds, blind=TRUE); M <- assay(vsd)
blocks <- list("Core effectors"=..., "FOXO-dependent"=..., "Core machinery"=...,
               "CLEAR / lysosomal"=..., "TFEB family"=..., "Respiratory chain"=...)
z <- t(scale(t(M[genes, ])))     # written to results/tables/heatmap_vst_zscores.csv

In [ ]:
# --- Python --- figures/fig13_per_sample_heatmap.png
im = ax.imshow(Z, aspect='auto', cmap='RdBu_r', vmin=-2, vmax=2)
# ETC block mean z: vehicle -0.752 vs SW +0.752
# autophagy blocks mean z: vehicle +0.033 vs SW -0.033